In [7]:
import pandas as pd
import glob
import os


datasets = ['tax']

In [8]:
def get_best_dc_profiling_result(dataset, mode):

    
    # Construct the path to CSV files
    base_path = "main_csv_results/main_comparison"
    pattern = os.path.join(base_path, "dc_discovery*.csv")
    
    # Load all CSV files that start with dc_discovery
    csv_files = glob.glob(pattern)
    
    if not csv_files:
        return None
    
    # Combine all dataframes
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)
    
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Filter by dataset and mode if those columns exist
    if 'dataset' in combined_df.columns:
        combined_df = combined_df[combined_df['dataset'] == dataset]
    if 'mode' in combined_df.columns:
        combined_df = combined_df[combined_df['mode'] == mode]
    
    # Get the row with the best (maximum) violation_f1 (change the metric if needed)
    metric_column = 'violation_f1'
    if metric_column in combined_df.columns and len(combined_df) > 0:
        best_idx = combined_df[metric_column].idxmax()
        return combined_df.loc[best_idx]
    
    return None

def get_best_norma_document(dataset):
    # Construct the path to CSV files
    base_path = "main_csv_results/main_comparison"
    pattern = os.path.join(base_path, "norma_document*.csv")
    
    # Load all CSV files that start with norma_document
    csv_files = glob.glob(pattern)
    
    if not csv_files:
        return None
    
    # Combine all dataframes
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)
    
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Filter by dataset if that column exists
    if 'dataset' in combined_df.columns:
        combined_df = combined_df[combined_df['dataset'] == dataset]
    
    # Filter by successful status if that column exists
    if 'status' in combined_df.columns:
        combined_df = combined_df[combined_df['status'] == 'success']
    
    # Get the row with the best (maximum) violation_f1_aggregate
    metric_column = 'violation_f1_aggregate'
    if metric_column in combined_df.columns and len(combined_df) > 0:
        best_idx = combined_df[metric_column].idxmax()
        return combined_df.loc[best_idx]
    
    return None


# modes for norma sentence can be SENTENCE_DIRECT or SENTENCE_FILTERED
def get_best_norma_sentence(dataset, mode):
    # Construct the path to CSV files
    base_path = "main_csv_results/main_comparison"
    pattern = os.path.join(base_path, "norma_sentence*.csv")
    
    # Load all CSV files that start with norma_sentence
    csv_files = glob.glob(pattern)
    
    if not csv_files:
        return None
    
    # Combine all dataframes
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        dfs.append(df)
    
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Filter by dataset if that column exists
    if 'dataset' in combined_df.columns:
        combined_df = combined_df[combined_df['dataset'] == dataset]
    
    # Filter by mode if that column exists
    if 'mode' in combined_df.columns:
        combined_df = combined_df[combined_df['mode'] == mode]
    
    # Filter by successful status if that column exists
    if 'status' in combined_df.columns:
        combined_df = combined_df[combined_df['status'] == 'success']
    
    # Get the row with the best (maximum) violation_f1_aggregate
    metric_column = 'violation_f1_aggregate'
    if metric_column in combined_df.columns and len(combined_df) > 0:
        best_idx = combined_df[metric_column].idxmax()
        return combined_df.loc[best_idx]
    
    return None

In [9]:
# best_exact = get_best_dc_profiling_result('tax', 'exact')
# best_approx = get_best_dc_profiling_result('tax', 'approximate')

# best_norma_document = get_best_norma_document('tax')
# best_norma_sentence_direct = get_best_norma_sentence('tax', 'SENTENCE_DIRECT')
# best_norma_sentence_filtered = get_best_norma_sentence('tax', 'SENTENCE_FILTERED')

# print("Best Exact DC Profiling Result:")
# print(best_exact)

# print("\nBest Approximate DC Profiling Result:")
# print(best_approx)

# print("\nBest Norma Document Result:")
# print(best_norma_document)

# print("\nBest Norma Sentence Direct Result:")
# print(best_norma_sentence_direct)

# print("\nBest Norma Sentence Filtered Result:")
# print(best_norma_sentence_filtered)

In [10]:
def create_results_table(datasets=['tax'], precision=2):
    """
    Create a LaTeX booktabs table with results from all approaches.
    
    Parameters:
    - datasets: list of dataset names to include
    - precision: number of decimal places for metrics
    """
    
    rows = []
    
    for dataset in datasets:
        # DC Profiling results
        for mode in ['exact', 'approximate']:
            result = get_best_dc_profiling_result(dataset, mode)
            if result is not None:
                rows.append({
                    'approach': 'DC Profiling',
                    'variant': mode.capitalize(),
                    'dataset': dataset,
                    'constraint_f1': result.get('constraint_f1', None),
                    'constraint_precision': result.get('constraint_precision', None),
                    'constraint_recall': result.get('constraint_recall', None),
                    'violation_f1_agg': result.get('violation_f1', None),
                    'violation_precision_agg': result.get('violation_precision', None),
                    'violation_recall_agg': result.get('violation_recall', None),
                    'constraint_syntactic': None,
                    'violation_f1_per_constraint': None
                })
        
        # Norma Document results
        result = get_best_norma_document(dataset)
        if result is not None:
            rows.append({
                'approach': 'NORMA',
                'variant': 'Document',
                'dataset': dataset,
                'constraint_f1': result.get('constraint_discovery_f1', None),
                'constraint_precision': result.get('constraint_discovery_precision', None),
                'constraint_recall': result.get('constraint_discovery_recall', None),
                'violation_f1_agg': result.get('violation_f1_aggregate', None),
                'violation_precision_agg': result.get('violation_precision_aggregate', None),
                'violation_recall_agg': result.get('violation_recall_aggregate', None),
                'constraint_syntactic': None,
                'violation_f1_per_constraint': None
            })
        
        # Norma Sentence results
        for mode in ['SENTENCE_DIRECT', 'SENTENCE_FILTERED']:
            result = get_best_norma_sentence(dataset, mode)
            if result is not None:
                variant_name = 'Sentence Direct' if mode == 'SENTENCE_DIRECT' else 'Sentence Filtered'
                rows.append({
                    'approach': 'NORMA',
                    'variant': variant_name,
                    'dataset': dataset,
                    'constraint_f1': result.get('constraint_detection_f1', None),
                    'constraint_precision': result.get('constraint_detection_precision', None),
                    'constraint_recall': result.get('constraint_detection_recall', None),
                    'violation_f1_agg': result.get('violation_f1_aggregate', None),
                    'violation_precision_agg': result.get('violation_precision_aggregate', None),
                    'violation_recall_agg': result.get('violation_recall_aggregate', None),
                    'constraint_syntactic': result.get('constraint_syntactic_predicate_jaccard', None),
                    'violation_f1_per_constraint': result.get('violation_discovery_per_constraint_f1', None)
                })
    
    # Count rows per dataset for multirow
    dataset_row_counts = {}
    for row in rows:
        ds = row['dataset']
        dataset_row_counts[ds] = dataset_row_counts.get(ds, 0) + 1
    
    # Find best F1 scores per dataset for each metric group
    best_constraint_f1 = {}
    best_violation_f1 = {}
    for row in rows:
        ds = row['dataset']
        if ds not in best_constraint_f1:
            best_constraint_f1[ds] = -1
            best_violation_f1[ds] = -1
        if row['constraint_f1'] is not None and row['constraint_f1'] > best_constraint_f1[ds]:
            best_constraint_f1[ds] = row['constraint_f1']
        if row['violation_f1_agg'] is not None and row['violation_f1_agg'] > best_violation_f1[ds]:
            best_violation_f1[ds] = row['violation_f1_agg']
    
    # Generate LaTeX table
    latex = []
    latex.append(r"\begin{table*}[ht]")
    latex.append(r"\centering")
    latex.append(r"\small")
    latex.append(r"\begin{tabular}{lllccccccccc}")
    latex.append(r"\toprule")
    latex.append(r"\textbf{Dataset} & \textbf{Approach} & \textbf{Variant} & \multicolumn{3}{c}{\textbf{Constraint Discovery}} & \multicolumn{3}{c}{\textbf{Violation (Aggregate)}} & \multicolumn{2}{c}{\textbf{Sentence-Level}} \\")
    latex.append(r"\cmidrule(lr){4-6} \cmidrule(lr){7-9} \cmidrule(lr){10-11}")
    latex.append(r" & & & \textbf{F1} & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} & \textbf{Prec.} & \textbf{Rec.} & \textbf{Synt. Jacc.} & \textbf{Viol. F1 (PC)} \\")
    latex.append(r"\midrule")
    
    current_dataset = None
    current_approach = None
    dataset_first_row = {}
    
    # First pass: identify first row of each dataset
    for i, row in enumerate(rows):
        ds = row['dataset']
        if ds not in dataset_first_row:
            dataset_first_row[ds] = i
    
    for i, row in enumerate(rows):
        ds = row['dataset']
        
        # Dataset column with multirow
        if i == dataset_first_row[ds]:
            dataset_str = r"\multirow{" + str(dataset_row_counts[ds]) + r"}{*}{" + ds + "}"
            current_dataset = ds
            current_approach = None  # Reset approach for new dataset
        else:
            dataset_str = ""
        
        # Approach column
        if row['approach'] != current_approach:
            approach_str = row['approach']
            current_approach = row['approach']
        else:
            approach_str = ""
        
        # Helper function to format values
        def fmt(val, is_best=False):
            if val is None:
                return "-"
            formatted = f"{val:.{precision}f}"
            if is_best and val > 0:
                return r"\textbf{" + formatted + "}"
            return formatted
        
        # Check if this row has the best F1 scores
        is_best_constraint = (row['constraint_f1'] is not None and 
                              row['constraint_f1'] == best_constraint_f1[ds])
        is_best_violation = (row['violation_f1_agg'] is not None and 
                             row['violation_f1_agg'] == best_violation_f1[ds])
        
        latex.append(
            f"{dataset_str} & {approach_str} & {row['variant']} & "
            f"{fmt(row['constraint_f1'], is_best_constraint)} & "
            f"{fmt(row['constraint_precision'])} & "
            f"{fmt(row['constraint_recall'])} & "
            f"{fmt(row['violation_f1_agg'], is_best_violation)} & "
            f"{fmt(row['violation_precision_agg'])} & "
            f"{fmt(row['violation_recall_agg'])} & "
            f"{fmt(row['constraint_syntactic'])} & "
            f"{fmt(row['violation_f1_per_constraint'])} \\\\"
        )
        
        # Add midrule between datasets
        if i < len(rows) - 1 and rows[i+1]['dataset'] != ds:
            latex.append(r"\midrule")
    
    latex.append(r"\bottomrule")
    latex.append(r"\end{tabular}")
    latex.append(r"\caption{Comparison of constraint discovery and violation detection results across different approaches. Best F1 scores per dataset are shown in bold. Synt. Jacc. = Constraint Syntactic Predicate Jaccard, Viol. F1 (PC) = Violation F1 per Constraint.}")
    latex.append(r"\label{tab:results}")
    latex.append(r"\end{table*}")
    
    return "\n".join(latex)

In [11]:
# Generate and print the LaTeX table
latex_table = create_results_table(datasets=['tax'], precision=2)
print(latex_table)

\begin{table*}[ht]
\centering
\small
\begin{tabular}{lllccccccccc}
\toprule
\textbf{Dataset} & \textbf{Approach} & \textbf{Variant} & \multicolumn{3}{c}{\textbf{Constraint Discovery}} & \multicolumn{3}{c}{\textbf{Violation (Aggregate)}} & \multicolumn{2}{c}{\textbf{Sentence-Level}} \\
\cmidrule(lr){4-6} \cmidrule(lr){7-9} \cmidrule(lr){10-11}
 & & & \textbf{F1} & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} & \textbf{Prec.} & \textbf{Rec.} & \textbf{Synt. Jacc.} & \textbf{Viol. F1 (PC)} \\
\midrule
\multirow{5}{*}{tax} & DC Profiling & Exact & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & - & - \\
 &  & Approximate & 0.00 & 0.00 & 0.00 & 0.01 & 0.00 & 1.00 & - & - \\
 & NORMA & Document & 0.76 & 0.70 & 0.83 & \textbf{0.99} & 1.00 & 0.98 & - & - \\
 &  & Sentence Direct & 0.63 & 0.46 & 1.00 & \textbf{0.99} & 1.00 & 0.98 & 0.83 & 0.83 \\
 &  & Sentence Filtered & \textbf{0.83} & 0.83 & 0.83 & 0.88 & 1.00 & 0.79 & 0.67 & 0.67 \\
\bottomrule
\end{tabular}
\caption{Comparison of constraint disc

In [12]:
def create_three_stage_results_table(datasets=['tax'], precision=2):
    """
    Create a LaTeX table organizing results by the three evaluation stages.
    
    Stage 1: Constraint Detection (Prec, Rec, F1)
    Stage 2: Syntactic Quality (Exact Match, Pred Jaccard, Soft Jaccard)  
    Stage 3a: Per-Constraint Violation (Prec, Rec, F1)
    Stage 3b: Aggregate Violation (Prec, Rec, F1)
    """
    
    rows = []
    
    for dataset in datasets:
        # DC Profiling results - only has Stage 3b (aggregate violation metrics)
        for mode in ['exact', 'approximate']:
            result = get_best_dc_profiling_result(dataset, mode)
            if result is not None:
                rows.append({
                    'approach': 'DC Profiling',
                    'variant': mode.capitalize(),
                    'dataset': dataset,
                    # Stage 1: Detection (N/A for DC)
                    'detection_precision': None,
                    'detection_recall': None,
                    'detection_f1': None,
                    # Stage 2: Syntactic Quality (N/A for DC)
                    'syntactic_exact': None,
                    'syntactic_pred_jaccard': None,
                    'syntactic_soft_jaccard': None,
                    # Stage 3a: Per-Constraint (N/A for DC)
                    'per_constraint_precision': None,
                    'per_constraint_recall': None,
                    'per_constraint_f1': None,
                    # Stage 3b: Aggregate
                    'aggregate_precision': result.get('violation_precision', None),
                    'aggregate_recall': result.get('violation_recall', None),
                    'aggregate_f1': result.get('violation_f1', None),
                })
        
        # Norma Document results - only has Stage 3b
        result = get_best_norma_document(dataset)
        if result is not None:
            rows.append({
                'approach': 'NORMA',
                'variant': 'Document',
                'dataset': dataset,
                # Stage 1: Detection (N/A for Document)
                'detection_precision': None,
                'detection_recall': None,
                'detection_f1': None,
                # Stage 2: Syntactic Quality (N/A for Document)
                'syntactic_exact': None,
                'syntactic_pred_jaccard': None,
                'syntactic_soft_jaccard': None,
                # Stage 3a: Per-Constraint (N/A for Document)
                'per_constraint_precision': None,
                'per_constraint_recall': None,
                'per_constraint_f1': None,
                # Stage 3b: Aggregate
                'aggregate_precision': result.get('violation_precision_aggregate', None),
                'aggregate_recall': result.get('violation_recall_aggregate', None),
                'aggregate_f1': result.get('violation_f1_aggregate', None),
            })
        
        # Norma Sentence results - has all stages
        for mode in ['SENTENCE_DIRECT', 'SENTENCE_FILTERED']:
            result = get_best_norma_sentence(dataset, mode)
            if result is not None:
                variant_name = 'Sentence Direct' if mode == 'SENTENCE_DIRECT' else 'Sentence Filtered'
                rows.append({
                    'approach': 'NORMA',
                    'variant': variant_name,
                    'dataset': dataset,
                    # Stage 1: Detection
                    'detection_precision': result.get('constraint_detection_precision', None),
                    'detection_recall': result.get('constraint_detection_recall', None),
                    'detection_f1': result.get('constraint_detection_f1', None),
                    # Stage 2: Syntactic Quality
                    'syntactic_exact': result.get('constraint_syntactic_exact_match', None),
                    'syntactic_pred_jaccard': result.get('constraint_syntactic_predicate_jaccard', None),
                    'syntactic_soft_jaccard': result.get('constraint_syntactic_soft_jaccard', None),
                    # Stage 3a: Per-Constraint
                    'per_constraint_precision': result.get('violation_discovery_per_constraint_precision', None),
                    'per_constraint_recall': result.get('violation_discovery_per_constraint_recall', None),
                    'per_constraint_f1': result.get('violation_discovery_per_constraint_f1', None),
                    # Stage 3b: Aggregate
                    'aggregate_precision': result.get('violation_precision_aggregate', None),
                    'aggregate_recall': result.get('violation_recall_aggregate', None),
                    'aggregate_f1': result.get('violation_f1_aggregate', None),
                })
    
    # Count rows per dataset for multirow
    dataset_row_counts = {}
    for row in rows:
        ds = row['dataset']
        dataset_row_counts[ds] = dataset_row_counts.get(ds, 0) + 1
    
    # Find best F1 scores per dataset for each stage
    best_detection_f1 = {}
    best_per_constraint_f1 = {}
    best_aggregate_f1 = {}
    
    for row in rows:
        ds = row['dataset']
        if ds not in best_detection_f1:
            best_detection_f1[ds] = -1
            best_per_constraint_f1[ds] = -1
            best_aggregate_f1[ds] = -1
        
        if row['detection_f1'] is not None and row['detection_f1'] > best_detection_f1[ds]:
            best_detection_f1[ds] = row['detection_f1']
        if row['per_constraint_f1'] is not None and row['per_constraint_f1'] > best_per_constraint_f1[ds]:
            best_per_constraint_f1[ds] = row['per_constraint_f1']
        if row['aggregate_f1'] is not None and row['aggregate_f1'] > best_aggregate_f1[ds]:
            best_aggregate_f1[ds] = row['aggregate_f1']
    
    # Generate LaTeX table
    latex = []
    latex.append(r"\begin{table*}[ht]")
    latex.append(r"\centering")
    latex.append(r"\scriptsize")
    latex.append(r"\begin{tabular}{lllcccccccccccc}")
    latex.append(r"\toprule")
    latex.append(r"\textbf{Dataset} & \textbf{Approach} & \textbf{Variant} & \multicolumn{3}{c}{\textbf{Stage 1: Detection}} & \multicolumn{3}{c}{\textbf{Stage 2: Syntactic}} & \multicolumn{3}{c}{\textbf{Stage 3a: Per-Const.}} & \multicolumn{3}{c}{\textbf{Stage 3b: Aggregate}} \\")
    latex.append(r"\cmidrule(lr){4-6} \cmidrule(lr){7-9} \cmidrule(lr){10-12} \cmidrule(lr){13-15}")
    latex.append(r" & & & \textbf{P} & \textbf{R} & \textbf{F1} & \textbf{EM} & \textbf{PJ} & \textbf{SJ} & \textbf{P} & \textbf{R} & \textbf{F1} & \textbf{P} & \textbf{R} & \textbf{F1} \\")
    latex.append(r"\midrule")
    
    current_dataset = None
    current_approach = None
    dataset_first_row = {}
    
    # First pass: identify first row of each dataset
    for i, row in enumerate(rows):
        ds = row['dataset']
        if ds not in dataset_first_row:
            dataset_first_row[ds] = i
    
    # Helper function to format values
    def fmt(val, is_best=False):
        if val is None:
            return "-"
        formatted = f"{val:.{precision}f}"
        if is_best and val > 0:
            return r"\textbf{" + formatted + "}"
        return formatted
    
    for i, row in enumerate(rows):
        ds = row['dataset']
        
        # Dataset column with multirow
        if i == dataset_first_row[ds]:
            dataset_str = r"\multirow{" + str(dataset_row_counts[ds]) + r"}{*}{" + ds + "}"
            current_dataset = ds
            current_approach = None  # Reset approach for new dataset
        else:
            dataset_str = ""
        
        # Approach column
        if row['approach'] != current_approach:
            approach_str = row['approach']
            current_approach = row['approach']
        else:
            approach_str = ""
        
        # Check if this row has the best F1 scores
        is_best_detection = (row['detection_f1'] is not None and 
                            row['detection_f1'] == best_detection_f1[ds])
        is_best_per_constraint = (row['per_constraint_f1'] is not None and 
                                  row['per_constraint_f1'] == best_per_constraint_f1[ds])
        is_best_aggregate = (row['aggregate_f1'] is not None and 
                            row['aggregate_f1'] == best_aggregate_f1[ds])
        
        latex.append(
            f"{dataset_str} & {approach_str} & {row['variant']} & "
            # Stage 1: Detection
            f"{fmt(row['detection_precision'])} & "
            f"{fmt(row['detection_recall'])} & "
            f"{fmt(row['detection_f1'], is_best_detection)} & "
            # Stage 2: Syntactic
            f"{fmt(row['syntactic_exact'])} & "
            f"{fmt(row['syntactic_pred_jaccard'])} & "
            f"{fmt(row['syntactic_soft_jaccard'])} & "
            # Stage 3a: Per-Constraint
            f"{fmt(row['per_constraint_precision'])} & "
            f"{fmt(row['per_constraint_recall'])} & "
            f"{fmt(row['per_constraint_f1'], is_best_per_constraint)} & "
            # Stage 3b: Aggregate
            f"{fmt(row['aggregate_precision'])} & "
            f"{fmt(row['aggregate_recall'])} & "
            f"{fmt(row['aggregate_f1'], is_best_aggregate)} \\\\"
        )
        
        # Add midrule between datasets
        if i < len(rows) - 1 and rows[i+1]['dataset'] != ds:
            latex.append(r"\midrule")
    
    latex.append(r"\bottomrule")
    latex.append(r"\end{tabular}")
    latex.append(r"\caption{Three-Stage Evaluation Results. Stage 1 (Detection): P/R/F1 for identifying constraint-bearing sentences (sentence+classification only). Stage 2 (Syntactic): Exact Match (EM), Predicate Jaccard (PJ), Soft Jaccard (SJ) for constraint syntax (sentence-based only). Stage 3a (Per-Constraint): P/R/F1 for individual constraint violation finding (sentence-based only). Stage 3b (Aggregate): P/R/F1 for pooled violation discovery. Best F1 scores per stage shown in bold.}")
    latex.append(r"\label{tab:three_stage_results}")
    latex.append(r"\end{table*}")
    
    return "\n".join(latex)

# Generate and print the three-stage results table
three_stage_table = create_three_stage_results_table(datasets=['tax'], precision=2)
print(three_stage_table)

\begin{table*}[ht]
\centering
\scriptsize
\begin{tabular}{lllcccccccccccc}
\toprule
\textbf{Dataset} & \textbf{Approach} & \textbf{Variant} & \multicolumn{3}{c}{\textbf{Stage 1: Detection}} & \multicolumn{3}{c}{\textbf{Stage 2: Syntactic}} & \multicolumn{3}{c}{\textbf{Stage 3a: Per-Const.}} & \multicolumn{3}{c}{\textbf{Stage 3b: Aggregate}} \\
\cmidrule(lr){4-6} \cmidrule(lr){7-9} \cmidrule(lr){10-12} \cmidrule(lr){13-15}
 & & & \textbf{P} & \textbf{R} & \textbf{F1} & \textbf{EM} & \textbf{PJ} & \textbf{SJ} & \textbf{P} & \textbf{R} & \textbf{F1} & \textbf{P} & \textbf{R} & \textbf{F1} \\
\midrule
\multirow{5}{*}{tax} & DC Profiling & Exact & - & - & - & - & - & - & - & - & - & 0.00 & 0.00 & 0.00 \\
 &  & Approximate & - & - & - & - & - & - & - & - & - & 0.00 & 1.00 & 0.01 \\
 & NORMA & Document & - & - & - & - & - & - & - & - & - & 1.00 & 0.98 & \textbf{0.99} \\
 &  & Sentence Direct & 0.46 & 1.00 & 0.63 & 0.83 & 0.83 & 0.92 & 0.83 & 0.83 & \textbf{0.83} & 1.00 & 0.98 & \textbf{0.99} 